# Episode 2 — Document Ingestion, Cleaning & Chunking

**Course:** Production RAG — YouTube Series  
**GitHub branch:** `episode/02`

> *"Your chunking strategy determines 80% of your RAG quality. Most people never tune it."*

## What we build in this episode

| Component | File | What it does |
|-----------|------|--------------|
| Cleaner | `src/rag/ingestion/cleaner.py` | 8-step text normalisation pipeline |
| Chunker | `src/rag/ingestion/chunker.py` | 4 chunking strategies with stats |
| Script  | `scripts/ingest.py` | Full CLI pipeline with progress bars |

## Episode roadmap
1. Load our 6 DHS PDFs and inspect raw artefacts
2. Walk through each cleaning step — before/after on live text
3. Run all 4 chunking strategies and compare with stats
4. Inspect metadata on a real chunk
5. Introduce the parent-child concept
6. Run the full ingestion pipeline on all 1466 pages
7. Set up the RAGAS baseline dataset (first 30 questions)

## 0. Setup

In [ ]:
import subprocess, sys
from pathlib import Path

# Install anything missing
pkgs = ["python-dotenv", "langchain", "langchain-openai", "langchain-community",
        "langchain-text-splitters", "pymupdf", "pdfplumber", "openai",
        "pydantic-settings", "rich", "tenacity"]
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet"] + pkgs, check=True)
print('✅ packages ready')

# Path setup
cwd       = Path().resolve()
repo_root = cwd.parent if cwd.name == 'notebooks' else cwd
src_path  = repo_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# Load env
from dotenv import load_dotenv
load_dotenv(repo_root / '.env', override=True)

print(f'Repo root: {repo_root}')

In [ ]:
# Load all pages with metadata (force reload every time)
import json
from rag.ingestion.loader import load_directory

data_dir     = repo_root / 'data' / 'raw'
meta_file    = repo_root / 'data' / 'metadata.json'
metadata_map = json.loads(meta_file.read_text())

all_pages = load_directory(data_dir, metadata_map=metadata_map)

from collections import Counter
by_country = Counter(p.country for p in all_pages)

print(f'Total pages: {len(all_pages)}')
print()
print(f'{"Country":<15} {"Pages":>6}')
print('-' * 23)
for country, n in sorted(by_country.items(), key=lambda x: -x[1]):
    label = country if country else '(no metadata)'
    print(f'{label:<15} {n:>6}')

## 1. Raw text artefacts — the problem

Before we clean anything, let's see exactly what we're dealing with.
This is the key motivation for the entire cleaning pipeline.

In [ ]:
# Pick a representative page from each country — middle of the document
# so we get real data content, not the cover page

samples = {}
for country in ['Nigeria', 'Kenya', 'Ghana', 'Ethiopia']:
    country_pages = [p for p in all_pages if p.country == country]
    if country_pages:
        # 1/3 through — past front matter, into real content
        idx = len(country_pages) // 3
        samples[country] = country_pages[idx]

print('Sample pages selected:')
for country, page in samples.items():
    print(f'  {country:<12} → {page.file_name} page {page.page_number}/{page.total_pages}  ({len(page.text):,} chars)')

In [ ]:
# Show raw text from Nigeria — our primary demo document
# Point out each artefact as you talk through it on camera

nigeria_raw = samples['Nigeria']
print(f'Nigeria DHS — Page {nigeria_raw.page_number} — RAW TEXT')
print('=' * 60)
print(nigeria_raw.text[:3000])
print('...')
print()
print('ARTEFACTS TO SPOT:')
print('  [ ] Unicode ligatures     — look for ﬁ, ﬂ')
print('  [ ] Hyphenated linebreaks — word-\nbreak')
print('  [ ] Footnote numbers      — "rate12 was" or "finding1,2"')
print('  [ ] Repeated headers      — ALL CAPS lines at top')
print('  [ ] Lone page numbers     — single digit on its own line')
print('  [ ] Excessive blank lines — 3+ consecutive newlines')

## 2. The cleaning pipeline — step by step

We apply 8 cleaning steps in sequence. Let's run each one individually
so you can see exactly what it changes — this is the core teaching moment.

In [ ]:
from rag.ingestion.cleaner import step_by_step_clean, PIPELINE_STEPS

# Run step-by-step on the Nigeria sample page
results = step_by_step_clean(nigeria_raw.text)

print(f'{'Step':<28} {'Before':>8} {'After':>8} {'Delta':>8}')
print('-' * 56)
for r in results:
    delta_str = f"{r['delta']:+}" if r['delta'] != 0 else '    —'
    print(f"{r['step']:<28} {r['chars_before']:>8,} {r['chars_after']:>8,} {delta_str:>8}")

total_before = len(nigeria_raw.text)
total_after  = results[-1]['chars_after']
print('-' * 56)
print(f"{'TOTAL REDUCTION':<28} {total_before:>8,} {total_after:>8,} {total_after - total_before:>+8,}")
print(f"\n  Noise removed: {(1 - total_after/total_before)*100:.1f}%")

In [ ]:
# Inspect each step's output individually
# On camera: walk through this cell pausing on each step to explain the regex

step_name = 'Unicode normalisation'   # change this to explore other steps

r = next(r for r in results if r['step'] == step_name)
print(f'Step: {r["step"]}')
print(f'Chars: {r["chars_before"]:,} → {r["chars_after"]:,}  (delta {r["delta"]:+})')
print()
print('Text after this step (first 500 chars):')
print(r['text_after'])

In [ ]:
# Run the FULL cleaning pipeline — single call
from rag.ingestion.cleaner import clean_page, clean_pages, cleaning_report

nigeria_cleaned = clean_page(nigeria_raw)
report = cleaning_report(nigeria_raw, nigeria_cleaned)

print('=== CLEANING REPORT ===')
for k, v in report.items():
    if 'preview' not in k:
        print(f'  {k:<20} {v}')

print()
print('=== BEFORE (first 600 chars) ===')
print(report['raw_preview'])
print()
print('=== AFTER  (first 600 chars) ===')
print(report['cleaned_preview'])

In [ ]:
# Clean ALL 1466 pages and see what gets filtered
# On camera: this is where you explain min_chars=80 and why we filter

print('Cleaning all pages (this takes ~10 seconds)...')
all_cleaned = clean_pages(all_pages, min_chars=80)

filtered = len(all_pages) - len(all_cleaned)
print(f'\nOriginal: {len(all_pages):,} pages')
print(f'Kept:     {len(all_cleaned):,} pages')
print(f'Filtered: {filtered:,} pages ({100*filtered/len(all_pages):.1f}%)')
print()
print('Filtered pages are: cover pages, blank pages,')
print('table-of-contents dividers, section header pages.')
print('Keeping them would dilute retrieval quality.')

In [ ]:
# Compare char counts before and after cleaning by country

from collections import defaultdict

raw_chars     = defaultdict(int)
cleaned_chars = defaultdict(int)

for p in all_pages:
    raw_chars[p.country] += len(p.text)

for p in all_cleaned:
    cleaned_chars[p.country] += len(p.text)

print(f'{"Country":<15} {"Raw chars":>12} {"Clean chars":>12} {"Reduction":>10}')
print('-' * 52)
for country in sorted(raw_chars):
    rc  = raw_chars[country]
    cc  = cleaned_chars.get(country, 0)
    pct = 100 * (1 - cc / rc) if rc else 0
    label = country if country else '(unknown)'
    print(f'{label:<15} {rc:>12,} {cc:>12,} {pct:>9.1f}%')

## 3. Chunking strategies — the comparison

This is the central teaching moment of Episode 2.
We run 4 strategies on the same dataset and measure the results.

**Key insight:** The numbers don't lie — but you also need to read the chunks.
A low `short_chunks` count and reasonable `avg_chars` are necessary but not sufficient.
You need to actually read sample chunks and check if sentences are complete.

In [ ]:
# Use Nigeria only for the strategy comparison — faster iteration
# Full corpus comparison comes in the next cell

from rag.ingestion.chunker import ChunkStrategy, chunk_pages, compare_strategies

nigeria_pages   = [p for p in all_cleaned if p.country == 'Nigeria']
print(f'Nigeria cleaned pages: {len(nigeria_pages)}')
print()

print('Running 4 chunking strategies on Nigeria DHS (chunk_size=800, overlap=150)...')

r_fixed     = chunk_pages(nigeria_pages, ChunkStrategy.FIXED,     chunk_size=800, chunk_overlap=150, return_result=True)
r_recursive = chunk_pages(nigeria_pages, ChunkStrategy.RECURSIVE, chunk_size=800, chunk_overlap=150, return_result=True)
r_sentence  = chunk_pages(nigeria_pages, ChunkStrategy.SENTENCE,  chunk_size=800, chunk_overlap=150, return_result=True)

print()
compare_strategies([r_fixed, r_recursive, r_sentence])

In [ ]:
# VISUAL: Show a mid-sentence split from FIXED vs clean split from RECURSIVE
# This is the on-camera demonstration that makes the case for recursive

print('=== FIXED chunking — chunk #5 ===')
print(r_fixed.documents[5].page_content)
print()
print('=== RECURSIVE chunking — chunk #5 ===')
print(r_recursive.documents[5].page_content)
print()
print('Notice: FIXED often cuts mid-sentence.')
print('RECURSIVE preserves complete thoughts — critical for health statistics.')

In [ ]:
# Check the Ethiopia report separately — it is narrative-heavy (mini report)
# RECURSIVE vs SENTENCE — which handles long paragraphs better?

ethiopia_pages = [p for p in all_cleaned if p.country == 'Ethiopia']
print(f'Ethiopia cleaned pages: {len(ethiopia_pages)}')
print()

r_eth_recursive = chunk_pages(ethiopia_pages, ChunkStrategy.RECURSIVE, chunk_size=800, chunk_overlap=150, return_result=True)
r_eth_sentence  = chunk_pages(ethiopia_pages, ChunkStrategy.SENTENCE,  chunk_size=800, chunk_overlap=150, return_result=True)

print('Ethiopia DHS mini-report — strategy comparison:')
compare_strategies([r_eth_recursive, r_eth_sentence])
print()
print('Note: The Ethiopia mini-report (FR363) is narrative-heavy.')
print('SENTENCE strategy often produces more uniform chunks for this format.')

In [ ]:
# FULL CORPUS comparison — all 4 countries, all strategies
# This is the "money shot" table for the episode

print('=== FULL CORPUS — All countries, all strategies ===')
print(f'(chunk_size=800, overlap=150, {len(all_cleaned):,} cleaned pages)\n')

r_all_fixed     = chunk_pages(all_cleaned, ChunkStrategy.FIXED,     chunk_size=800, chunk_overlap=150, return_result=True)
r_all_recursive = chunk_pages(all_cleaned, ChunkStrategy.RECURSIVE, chunk_size=800, chunk_overlap=150, return_result=True)
r_all_sentence  = chunk_pages(all_cleaned, ChunkStrategy.SENTENCE,  chunk_size=800, chunk_overlap=150, return_result=True)

compare_strategies([r_all_fixed, r_all_recursive, r_all_sentence])

print()
print('DECISION: We use RECURSIVE for the rest of the course.')
print('Rationale: lowest short_chunks, cleanest sentence boundaries,')
print('best balance between avg_chars and total_chunks.')

In [ ]:
# CHUNK SIZE sensitivity analysis — does 800 vs 600 vs 1000 matter?
# On camera: "The most common question I get is: what chunk size should I use?"

print('Chunk size sensitivity (RECURSIVE, Nigeria only):\n')
print(f'{"chunk_size":<12} {"Chunks":>7} {"Avg":>6} {"Short":>6} {"Long":>6}')
print('-' * 40)

for size in [400, 600, 800, 1000, 1200]:
    r = chunk_pages(nigeria_pages, ChunkStrategy.RECURSIVE, chunk_size=size, chunk_overlap=size//5, return_result=True)
    s = r.stats
    print(f'{size:<12} {s["total_chunks"]:>7,} {s["avg_chars"]:>6} {s["short_chunks"]:>6} {s["long_chunks"]:>6}')

print()
print('Rule of thumb: chunk_size ≈ 600–900 chars for DHS-style documents.')
print('Too small → too many chunks, retrieval noise.')
print('Too large → chunks exceed LLM context window; precision drops.')
print('We use 800 as our default. Episode 9 will measure this with RAGAS.')

## 4. Metadata — what travels with every chunk

In [ ]:
# Inspect 3 chunks from different countries — show metadata variety
# This is where you explain WHY metadata matters for filtered retrieval

final_chunks = r_all_recursive.documents

# Pick one chunk per country
seen_countries = set()
showcase = []
for doc in final_chunks:
    c = doc.metadata.get('country', '')
    if c and c not in seen_countries:
        showcase.append(doc)
        seen_countries.add(c)
    if len(showcase) == 4:
        break

for i, doc in enumerate(showcase, 1):
    print(f'━━━ Chunk {i} ━━━')
    print(f'CONTENT: {doc.page_content[:200]}...')
    print('METADATA:')
    for k, v in doc.metadata.items():
        print(f'  {k:<22} {repr(v)}')
    print()

In [ ]:
# Demonstrate what metadata filtering enables — preview of Episode 7
# On camera: "This metadata is why we can answer 'Show me Kenya data from 2022'"

# Filter by country
kenya_chunks   = [d for d in final_chunks if d.metadata.get('country') == 'Kenya']
nigeria_chunks = [d for d in final_chunks if d.metadata.get('country') == 'Nigeria']
ghana_chunks   = [d for d in final_chunks if d.metadata.get('country') == 'Ghana']

print('Chunks available per country filter:')
print(f'  Kenya:    {len(kenya_chunks):,} chunks')
print(f'  Nigeria:  {len(nigeria_chunks):,} chunks')
print(f'  Ghana:    {len(ghana_chunks):,} chunks')
print(f'  Ethiopia: {len([d for d in final_chunks if d.metadata.get("country") == "Ethiopia"]):,} chunks')
print(f'  All:      {len(final_chunks):,} chunks')
print()
print('Episode 7 shows how to extract these filters automatically from the query:')
print('  "What does the Kenya report say about maternal mortality?"')
print('  → LLM extracts: country="Kenya"')
print('  → pgvector searches only the 3,xxx Kenya chunks')
print('  → Retrieval precision dramatically improves')

## 5. The parent-child concept — preview of Episode 15

In [ ]:
# Parent-child chunking — conceptual demonstration
# On camera: draw this on a whiteboard: [Parent 1600 chars] → [Child][Child][Child][Child]

from rag.ingestion.chunker import build_parent_child_pairs

# Use a small subset so this is fast
sample_pages = [p for p in all_cleaned if p.country == 'Nigeria'][:20]

pairs = build_parent_child_pairs(
    sample_pages,
    parent_size=1600,
    child_size=400,
    overlap=50,
)

total_children = sum(len(p.children) for p in pairs)
print(f'From {len(sample_pages)} pages:')
print(f'  {len(pairs)} parent chunks  (~1600 chars each)')
print(f'  {total_children} child chunks   (~400 chars each)')
print(f'  avg {total_children/len(pairs):.1f} children per parent')
print()

# Show a parent-child pair
demo_pair = pairs[3]   # pick one with multiple children
print('━━━ PARENT CHUNK ━━━')
print(f'Length: {len(demo_pair.parent.page_content)} chars')
print(demo_pair.parent.page_content[:500])
print('...')
print()
print(f'━━━ CHILD CHUNKS ({len(demo_pair.children)} children) ━━━')
for i, child in enumerate(demo_pair.children[:3]):
    print(f'Child {i+1} ({len(child.page_content)} chars) | parent_id: {child.metadata["parent_id"][:20]}...')
    print(child.page_content[:200])
    print()

In [ ]:
# The retrieval flow with parent-child:
# On camera: explain this step by step

print('Parent-Child Retrieval Flow:')
print()
print('  1. INDEX TIME:   Child chunks (400 chars) → embedded → stored in pgvector')
print('                   Parent chunks (1600 chars) → stored by parent_id')
print()
print('  2. QUERY TIME:   User asks a question')
print('                   Vector search finds the most relevant CHILD chunks')
print('                   (small chunks = better precision)')
print()
print('  3. GENERATION:   Look up each child\'s parent_id')
print('                   Pass the PARENT chunk to the LLM')
print('                   (large chunk = more context for generation)')
print()
print('Result: Precision of small chunks + context richness of large chunks.')
print('Full implementation in Episode 15.')

## 6. Full ingestion pipeline — all 1466 pages

This runs the complete pipeline:
`load_directory → clean_pages → chunk_pages → (ready for pgvector)`

We don't embed or index in this episode (Episode 4 covers pgvector).
Here we just measure what the complete input to the embedder will look like.

In [ ]:
# Run the full pipeline and measure the output
import time

t0 = time.perf_counter()

# Step 1: Clean (already done above, reuse)
# Step 2: Chunk with our chosen strategy
final_docs = chunk_pages(
    all_cleaned,
    strategy=ChunkStrategy.RECURSIVE,
    chunk_size=800,
    chunk_overlap=150,
)

elapsed = time.perf_counter() - t0

from rag.ingestion.chunker import chunk_stats
stats = chunk_stats(final_docs)

print(f'Full pipeline complete in {elapsed:.1f}s')
print()
print('=== FINAL CORPUS STATISTICS ===')
print(f'  Input pages:    {len(all_pages):,}')
print(f'  After cleaning: {len(all_cleaned):,} pages  ({100*(len(all_pages)-len(all_cleaned))/len(all_pages):.1f}% filtered)')
print(f'  Output chunks:  {stats["total_chunks"]:,}')
print(f'  Avg chunk size: {stats["avg_chars"]} chars')
print(f'  Median:         {stats["median_chars"]} chars')
print(f'  Std deviation:  {stats["std_chars"]} chars')
print(f'  Range:          {stats["min_chars"]}–{stats["max_chars"]} chars')
print(f'  Short (<100):   {stats["short_chunks"]} chunks  (noise candidates)')
print(f'  Long (>1200):   {stats["long_chunks"]} chunks  (context risk)')
print(f'  Total content:  {stats["total_chars"]/1e6:.2f}M chars')

In [ ]:
# Chunks per country — final distribution
from collections import Counter

by_country = Counter(d.metadata.get('country', 'unknown') for d in final_docs)

print('Final chunks per country:')
print()
print(f'{"Country":<15} {"Chunks":>8} {"Share":>8}')
print('-' * 33)
for country, n in sorted(by_country.items(), key=lambda x: -x[1]):
    pct = 100 * n / len(final_docs)
    bar = '█' * (n // 200)
    print(f'{country:<15} {n:>8,} {pct:>7.1f}%  {bar}')

In [ ]:
# Embedding cost estimate — important for budgeting
# On camera: "Before you embed 10,000 chunks, know what it costs."

total_chars  = stats['total_chars']
# OpenAI text-embedding-3-small: ~4 chars per token, $0.02 per 1M tokens
est_tokens   = total_chars / 4
cost_per_1m  = 0.02   # USD
est_cost     = (est_tokens / 1_000_000) * cost_per_1m

print('Embedding cost estimate (text-embedding-3-small):')
print(f'  Total chars:    {total_chars:,}')
print(f'  Est. tokens:    {est_tokens:,.0f}  (chars ÷ 4)')
print(f'  Cost per 1M:    ${cost_per_1m:.3f}')
print(f'  Est. total:     ${est_cost:.4f}  ← less than a cent')
print()
print('Verdict: Embed everything. The cost is negligible for this corpus.')
print('At 10x corpus size it is still under $0.10.')
print('ONNX local embeddings cost $0.00 — covered in Episode 3.')

## 7. RAGAS baseline test set — 30 questions

We set this up now so it's ready for Episode 9.
These are questions with known answers from our actual PDFs.
We'll score against these 30 questions after every phase.

In [ ]:
# Save the test set to the evaluation directory
import json
from pathlib import Path

test_questions = [
    # ── Single-hop factual ────────────────────────────────────────────────────
    {"id": "Q01", "question": "What is the total fertility rate in Nigeria according to the 2021 DHS report?",
     "country": "Nigeria", "year": "2021", "type": "factual"},
    {"id": "Q02", "question": "What percentage of births in Kenya are attended by skilled health personnel according to the 2022 DHS?",
     "country": "Kenya", "year": "2022", "type": "factual"},
    {"id": "Q03", "question": "What is the under-5 mortality rate per 1,000 live births in Ghana according to the 2022 DHS?",
     "country": "Ghana", "year": "2022", "type": "factual"},
    {"id": "Q04", "question": "What proportion of women in Ethiopia have received antenatal care from a skilled provider?",
     "country": "Ethiopia", "year": "2019", "type": "factual"},
    {"id": "Q05", "question": "What is the contraceptive prevalence rate among married women in Nigeria?",
     "country": "Nigeria", "year": "2021", "type": "factual"},
    {"id": "Q06", "question": "What percentage of women in Kenya have secondary or higher education?",
     "country": "Kenya", "year": "2022", "type": "factual"},
    {"id": "Q07", "question": "What is the neonatal mortality rate in Ghana?",
     "country": "Ghana", "year": "2022", "type": "factual"},
    {"id": "Q08", "question": "What proportion of children under 5 in Ethiopia are stunted?",
     "country": "Ethiopia", "year": "2019", "type": "factual"},
    {"id": "Q09", "question": "What is the median age at first marriage for women in Nigeria?",
     "country": "Nigeria", "year": "2021", "type": "factual"},
    {"id": "Q10", "question": "What percentage of women in Kenya use modern contraception?",
     "country": "Kenya", "year": "2022", "type": "factual"},

    # ── Regional / comparative ─────────────────────────────────────────────────
    {"id": "Q11", "question": "What are the regional differences in under-5 mortality within Nigeria?",
     "country": "Nigeria", "year": "2021", "type": "regional"},
    {"id": "Q12", "question": "How does contraceptive use vary between urban and rural women in Ghana?",
     "country": "Ghana", "year": "2022", "type": "regional"},
    {"id": "Q13", "question": "What are the county-level differences in maternal health outcomes in Kenya?",
     "country": "Kenya", "year": "2022", "type": "regional"},
    {"id": "Q14", "question": "How do child nutrition indicators differ between regions in Ethiopia?",
     "country": "Ethiopia", "year": "2019", "type": "regional"},
    {"id": "Q15", "question": "What is the difference in skilled birth attendance between the richest and poorest wealth quintiles in Nigeria?",
     "country": "Nigeria", "year": "2021", "type": "regional"},

    # ── Education-health relationship ──────────────────────────────────────────
    {"id": "Q16", "question": "How does women's education level relate to contraceptive use in Kenya?",
     "country": "Kenya", "year": "2022", "type": "relational"},
    {"id": "Q17", "question": "What is the relationship between mother's education and child stunting in Ethiopia?",
     "country": "Ethiopia", "year": "2019", "type": "relational"},
    {"id": "Q18", "question": "How does household wealth affect skilled birth attendance in Ghana?",
     "country": "Ghana", "year": "2022", "type": "relational"},
    {"id": "Q19", "question": "What is the association between women's decision-making power and health service use in Nigeria?",
     "country": "Nigeria", "year": "2021", "type": "relational"},
    {"id": "Q20", "question": "How does place of delivery relate to neonatal mortality in Kenya?",
     "country": "Kenya", "year": "2022", "type": "relational"},

    # ── Multi-country (tests cross-corpus retrieval) ────────────────────────────
    {"id": "Q21", "question": "Compare the total fertility rate between Nigeria and Ghana.",
     "country": None, "year": None, "type": "multi_country"},
    {"id": "Q22", "question": "Which country in our dataset has the highest under-5 mortality rate?",
     "country": None, "year": None, "type": "multi_country"},
    {"id": "Q23", "question": "Compare contraceptive prevalence rates across all countries in the dataset.",
     "country": None, "year": None, "type": "multi_country"},
    {"id": "Q24", "question": "Which country has the highest proportion of women with secondary education?",
     "country": None, "year": None, "type": "multi_country"},
    {"id": "Q25", "question": "How do skilled birth attendance rates compare between Kenya and Ethiopia?",
     "country": None, "year": None, "type": "multi_country"},

    # ── Difficult / ambiguous (tests guardrails) ───────────────────────────────
    {"id": "Q26", "question": "What is the maternal mortality ratio in Nigeria?",
     "country": "Nigeria", "year": "2021", "type": "ambiguous",
     "note": "May not be in mini-report; tests 'I don't know' response"},
    {"id": "Q27", "question": "What is the trend in female education in West Africa over the past 20 years?",
     "country": None, "year": None, "type": "ambiguous",
     "note": "Trend requires multiple data points; may not be answerable from our corpus"},
    {"id": "Q28", "question": "What medication should a pregnant woman take to prevent malaria?",
     "country": None, "year": None, "type": "out_of_scope",
     "note": "Medical advice — guardrails should reject this"},
    {"id": "Q29", "question": "According to Kenya DHS 2022 Volume II, what are the fertility preferences of women?",
     "country": "Kenya", "year": "2022", "type": "specific_volume",
     "note": "Tests retrieval from FR380bis specifically"},
    {"id": "Q30", "question": "What corrections were made in the Kenya DHS 2022 erratum report?",
     "country": "Kenya", "year": "2022", "type": "erratum",
     "note": "Tests retrieval from FR380erratum specifically"},
]

# Save to the evaluation test_set directory
test_set_dir = repo_root / 'src' / 'rag' / 'evaluation' / 'test_set'
test_set_dir.mkdir(parents=True, exist_ok=True)

output_path = test_set_dir / 'questions.json'
with open(output_path, 'w') as f:
    json.dump(test_questions, f, indent=2)

print(f'✅ Saved {len(test_questions)} test questions to:')
print(f'   {output_path}')
print()
print('Question type breakdown:')
from collections import Counter
type_counts = Counter(q['type'] for q in test_questions)
for qtype, count in sorted(type_counts.items()):
    print(f'  {qtype:<20} {count}')

## 8. Run the CLI ingestion script

In the actual recording, show this running in the terminal — not as a notebook cell.
This is what viewers will copy and run themselves.

In [ ]:
# Show the commands viewers run in their terminal
print('Commands to run in your terminal:')
print()
print('# Preview without writing to DB:')
print('python scripts/ingest.py --dry-run')
print()
print('# Full ingestion with metadata file:')
print('python scripts/ingest.py --metadata data/metadata.json')
print()
print('# Try different chunking strategy:')
print('python scripts/ingest.py --strategy sentence --metadata data/metadata.json')
print()
print('# Use local ONNX embeddings (no API cost):')
print('python scripts/ingest.py --backend onnx --metadata data/metadata.json')
print()
print('# Or via Makefile:')
print('make ingest')

## ✅ Episode 2 complete

| Accomplishment | Detail |
|---------------|--------|
| 8-step cleaning pipeline | Built and tested on all 1,466 pages |
| 4 chunking strategies | Compared with full statistics table |
| Strategy decision | RECURSIVE selected — lowest noise, cleanest boundaries |
| Chunk size decision | 800 chars — balanced precision vs context |
| Parent-child concept | Introduced — full implementation in Episode 15 |
| Metadata design | country, year, report_type on every chunk |
| RAGAS test set | 30 questions saved to evaluation/test_set/ |
| Embedding cost | Estimated < $0.01 for full corpus |

## What's next — Episode 3

**Embeddings — turning text into meaning:**
- What is a vector embedding? Visual intuition without the maths
- OpenAI `text-embedding-3-small` — 1536 dimensions, $0.02/1M tokens
- Local ONNX with `all-MiniLM-L12-v2` — 768 dimensions, $0.00
- Batch embedding all chunks from Episode 2
- Cosine similarity demo: "maternal mortality" vs "mothers dying in childbirth"
- When to use cloud vs local embeddings

**GitHub branch:** `episode/03`

---
*Questions? Drop them in the YouTube comments or GitHub Discussions.*